# Neuron DSI: Signal Infinite Cascade (SIC) Simulation
This script implements a high-throughput simulation pipeline to calculate neuronal responses using the Signal Infinite Cascade (SIC) model. It maps the functional output of sensory circuits under Moving Edge stimuli to determine the Direction Selectivity Index (DSI).

In [1]:
import sys
import os

# -----------------------------
# DEPENDENCIES
# -----------------------------
sys.path.append(os.path.abspath('../util'))

from stimulus_Loader import StimulusProcessor
from SIC import SICModelTorch

In [2]:
import os
import numpy as np
import matplotlib.pyplot as plt
from tqdm import tqdm
from datetime import datetime
import csv
import glob
import torch

# ==================================================
# Result directory (npz)
# ==================================================
RESPONSE_ROOT = "../results/PRS/DSI"


# ==================================================
# Global npz deduplication (scan ALL response folders)
# ==================================================
print("Scanning existing npz files for global deduplication...")
existing_npz = set()

for npz_file in glob.glob(
    os.path.join(RESPONSE_ROOT, "**", "*.npz"),
    recursive=True
):
    base = os.path.splitext(os.path.basename(npz_file))[0]
    existing_npz.add(base)

print(f"Found {len(existing_npz)} existing response files")


# ---------------------------
# Initialize StimulusProcessor
# ---------------------------
print("\nInitializing StimulusProcessor...")
stimulus_processor = StimulusProcessor(
    target_dt_ms=1.0,
    target_size=(41, 82),
    is_visual=False,
    fps=1000.0
)


# ---------------------------
# Load stimulus videos (AFTER global dedup)
# ---------------------------
video_files = sorted(glob.glob("../stimulus/MovingEdge/*.npz"))

if not video_files:
    raise RuntimeError("❌ No .gif files found.")

print(f"Found {len(video_files)} video files")

stimulus_dataset = {}
skipped = 0

print("\nLoading stimuli (global npz dedup)...")
for v_file in tqdm(video_files, desc="Loading stimuli", unit="video"):

    stim_base = os.path.splitext(os.path.basename(v_file))[0]

    if stim_base in existing_npz:
        skipped += 1
        continue

    stimulus, _ = stimulus_processor.process_npz(
        v_file,
        verbose=False
    )

    key_name = os.path.splitext(v_file)[0]
    stimulus_dataset[key_name] = stimulus


print(f"✅ Stimuli loaded: {len(stimulus_dataset)}")
print(f"⏭️  Stimuli skipped (existing npz anywhere): {skipped}")


# ==================================================
# Initialize LMCs model (AFTER stimulus filtering)
# ==================================================
print("\nInitializing LMCs model...")
SIC_model = SICModelTorch(
    output_dir="../results/PRS/DSI",
    t_step=10,
    rate=100,
    device=torch.device("cuda:1")
)


# ---------------------------
# Load neuron IDs
# ---------------------------
def read_all_neuron_ids(filename='../data/classification.txt'):
    neuron_ids = []
    try:
        with open(filename, 'r') as f:
            reader = csv.DictReader(f)
            for row in reader:
                if 'root_id' in row and row['root_id'].strip():
                    try:
                        neuron_ids.append(int(row['root_id'].strip()))
                    except ValueError:
                        continue
    except FileNotFoundError:
        print(f"Error: File '{filename}' not found")
    except Exception as e:
        print(f"Error reading file: {str(e)}")
    return neuron_ids


print("\nLoading neuron weights...")
neuron_ids = read_all_neuron_ids()
print(f"Found {len(neuron_ids)} neurons")

weights = SIC_model.load_weights(neuron_ids,normalize=True) ## normalize weights to [-1,1]
print("Weights loaded successfully")


# ---------------------------
# Calculate responses
# ---------------------------
print("\nCalculating responses...")
for stim_name, stimulus in tqdm(
    stimulus_dataset.items(),
    desc="Processing responses",
    unit="stimulus"
):
    name = os.path.splitext(os.path.basename(stim_name))[0]

    SIC_model.calculate_response_baseline(
        stimulus,
        weights,
        responce_threshold=0,
        baseline_steps=50,
        stim_name=name
    )


# ---------------------------
# Summary
# ---------------------------
print("\n" + "=" * 50)
print("PREPROCESSING COMPLETE")
print("=" * 50)
print(f"Total video files found: {len(video_files)}")
print(f"Stimuli skipped (existing npz): {skipped}")
print(f"Stimuli processed: {len(stimulus_dataset)}")
print(f"Timestamp: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print("=" * 50)


Scanning existing npz files for global deduplication...
Found 0 existing response files

Initializing StimulusProcessor...
Found 120 video files

Loading stimuli (global npz dedup)...


Loading stimuli: 100%|██████████| 120/120 [00:23<00:00,  5.02video/s]


✅ Stimuli loaded: 120
⏭️  Stimuli skipped (existing npz anywhere): 0

Initializing LMCs model...
Using device: cuda:1

Loading neuron weights...
Found 139255 neurons
Weights loaded successfully

Calculating responses...


Processing responses: 100%|██████████| 120/120 [3:19:52<00:00, 99.93s/stimulus]  


PREPROCESSING COMPLETE
Total video files found: 120
Stimuli skipped (existing npz): 0
Stimuli processed: 120
Timestamp: 2026-06-15 23:55:59


In [ ]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-

import os
import glob
from pathlib import Path
from concurrent.futures import ProcessPoolExecutor, as_completed

import numpy as np
import pandas as pd
from tqdm import tqdm

responses_dir = "../results/PRS/DSI/202606"
cell_type_file = "../data/cell_type.txt"
classification_file = "../data/classification.txt"

out_dir = "../results/DSI"
side_out_dir = os.path.join(out_dir, "side_separated")

speeds = (2, 4, 8, 12, 16)
angles_deg = tuple(range(0, 360, 30))
EDGE_TYPES = ("ON", "OFF")

PEAK_THRESHOLD = 0.0
DIFF_THRESHOLD = 0

DEFAULT_LOAD_WORKERS = min(32, os.cpu_count() or 1)
NUM_LOAD_WORKERS = int(os.environ.get("NPZ_LOAD_WORKERS", DEFAULT_LOAD_WORKERS))

PEAK_DTYPE = np.float32


def parse_npz_name(file_path):
    base = Path(file_path).stem
    parts = base.split("_")
    try:
        edge_type = parts[0]
        angle = int([p for p in parts if p.startswith("angle")][0].replace("angle", ""))
        speed = int([p for p in parts if p.startswith("speed")][0].replace("speed", ""))
    except Exception:
        return None

    if edge_type not in EDGE_TYPES or speed not in speeds or angle not in angles_deg:
        return None
    return edge_type, speed, angle


def load_one_npz_peak(file_path):
    meta = parse_npz_name(file_path)
    if meta is None:
        return None

    edge_type, speed, angle = meta

    try:
        data = np.load(file_path, allow_pickle=False)
    except ValueError:
        data = np.load(file_path, allow_pickle=True)

    with data:
        neuron_ids = data["neuron_ids"].astype(str)
        responses = data["responses"]

        if responses.ndim == 1:
            peak_values = responses.astype(PEAK_DTYPE, copy=True)
        else:
            peak_values = responses.max(axis=1).astype(PEAK_DTYPE, copy=False)

        peak_values = np.asarray(peak_values, dtype=PEAK_DTYPE)
        peak_values[peak_values < 0] = 0.0

        if PEAK_THRESHOLD > 0:
            peak_values[peak_values < PEAK_THRESHOLD] = 0.0

    return edge_type, speed, angle, neuron_ids, peak_values


def condition_col_index(edge_type, speed, angle):
    ei = EDGE_TYPES.index(edge_type)
    si = speeds.index(speed)
    ai = angles_deg.index(angle)
    return ei * len(speeds) * len(angles_deg) + si * len(angles_deg) + ai


def build_peak_matrix(entries):
    valid_entries = [e for e in entries.values() if e is not None]
    if not valid_entries:
        raise RuntimeError("没有找到有效 npz，请检查 responses_dir 和文件名格式。")

    ref_ids = valid_entries[0]["neuron_ids"]
    same_order = True
    for e in valid_entries[1:]:
        ids = e["neuron_ids"]
        if len(ids) != len(ref_ids) or not np.array_equal(ids, ref_ids):
            same_order = False
            break

    n_conditions = len(EDGE_TYPES) * len(speeds) * len(angles_deg)

    if same_order:
        all_neurons = ref_ids.astype(str)
        data_matrix = np.zeros((len(all_neurons), n_conditions), dtype=PEAK_DTYPE)
        for (et, s, a), e in entries.items():
            if e is None:
                continue
            data_matrix[:, condition_col_index(et, s, a)] = e["peak_values"]
        return all_neurons, data_matrix

    print("检测到不同 npz 的 neuron_ids 顺序不完全一致，使用 union 映射填充。")
    all_neurons = np.unique(np.concatenate([e["neuron_ids"].astype(str) for e in valid_entries])).astype(str)
    nid2idx = {nid: i for i, nid in enumerate(all_neurons)}
    data_matrix = np.zeros((len(all_neurons), n_conditions), dtype=PEAK_DTYPE)

    for (et, s, a), e in tqdm(entries.items(), desc="Filling peak matrix"):
        if e is None:
            continue
        row_idx = np.fromiter((nid2idx[nid] for nid in e["neuron_ids"].astype(str)),
                              dtype=np.int64, count=len(e["neuron_ids"]))
        data_matrix[row_idx, condition_col_index(et, s, a)] = e["peak_values"]

    return all_neurons, data_matrix


def read_metadata(all_neurons):
    cell_type_df = pd.read_csv(
        cell_type_file,
        header=None,
        names=["root_id", "primary_type", "additional_type"],
        dtype={"root_id": str}
    ).set_index("root_id")

    classification_df = pd.read_csv(
        classification_file,
        dtype={"root_id": str}
    ).set_index("root_id")

    types = cell_type_df.reindex(all_neurons)["primary_type"].fillna("").to_numpy()
    sides = classification_df.reindex(all_neurons)["side"].fillna("").to_numpy()
    return cell_type_df, classification_df, types, sides


def save_wide_peak_csv(all_neurons, data_matrix, types, sides):
    cols_response = [
        f"{et}_edge_speed{s}_angle{a}"
        for et in EDGE_TYPES
        for s in speeds
        for a in angles_deg
    ]

    df_peak = pd.DataFrame(data_matrix, columns=cols_response)
    df_peak.insert(0, "side", sides)
    df_peak.insert(0, "type", types)
    df_peak.insert(0, "id", all_neurons)

    os.makedirs(out_dir, exist_ok=True)
    out_csv = os.path.join(out_dir, "peak_responses_all_wide.csv")
    df_peak.to_csv(out_csv, index=False)
    print(f"Peak responses CSV saved: {out_csv}")
    return df_peak


def save_min_removed_csv(df_peak, data_matrix):
    values = data_matrix.copy()
    n_angles = len(angles_deg)

    for et in EDGE_TYPES:
        for s in speeds:
            start = condition_col_index(et, s, angles_deg[0])
            stop = start + n_angles
            block = values[:, start:stop]

            nonzero_block = np.where(block > 0, block, np.inf)
            baseline = nonzero_block.min(axis=1)
            baseline[~np.isfinite(baseline)] = 0.0

            block -= baseline[:, None]
            np.maximum(block, 0.0, out=block)
            values[:, start:stop] = block

    response_cols = df_peak.columns[3:]
    df_out = df_peak[["id", "type", "side"]].copy()
    df_out = pd.DataFrame(
        values,
        columns=response_cols
    )

    out_csv = os.path.join(out_dir, "peak_responses_all_wide_min_removed.csv")
    df_out.to_csv(out_csv, index=False)
    print(f"Peak-min-removed CSV saved: {out_csv}")


def compute_dsi_for_edge_vectorized(edge_matrix, theta_complex, diff_threshold=0.01):
    r_min = edge_matrix.min(axis=2, keepdims=True)
    r_max = edge_matrix.max(axis=2)
    r_range = r_max - r_min[..., 0]

    delta = edge_matrix - r_min
    np.maximum(delta, 0.0, out=delta)

    keep_speed = r_range >= diff_threshold

    vec_speed = np.tensordot(delta, theta_complex, axes=([2], [0]))
    denom_speed = delta.sum(axis=2)

    dsi_speed = np.zeros(edge_matrix.shape[:2], dtype=np.float32)
    angle_speed = np.full(edge_matrix.shape[:2], -1.0, dtype=np.float32)

    valid_speed = keep_speed & (denom_speed > 1e-12)
    dsi_speed[valid_speed] = (np.abs(vec_speed[valid_speed]) / denom_speed[valid_speed]).astype(np.float32)
    dsi_speed[dsi_speed > 1.0] = 1.0 
    angle_speed[valid_speed] = (np.degrees(np.angle(vec_speed[valid_speed])) % 360).astype(np.float32)

    delta_all = delta.copy()
    delta_all[~keep_speed, :] = 0.05

    vec_all = np.tensordot(delta_all, theta_complex, axes=([2], [0])).sum(axis=1)
    denom_all = delta_all.sum(axis=(1, 2))

    dsi_all = np.zeros(edge_matrix.shape[0], dtype=np.float32)
    angle_all = np.full(edge_matrix.shape[0], -1.0, dtype=np.float32)

    valid_all = denom_all > 1e-12
    dsi_all[valid_all] = (np.abs(vec_all[valid_all]) / denom_all[valid_all]).astype(np.float32)
    dsi_all[dsi_all > 1.0] = 1.0
    angle_all[valid_all] = (np.degrees(np.angle(vec_all[valid_all])) % 360).astype(np.float32)

    return dsi_speed, angle_speed, dsi_all, angle_all


def save_dsi_csvs(all_neurons, data_matrix, types, sides):
    n_neurons = len(all_neurons)
    n_edges = len(EDGE_TYPES)
    n_speeds = len(speeds)
    n_angles = len(angles_deg)

    response_4d = data_matrix.reshape(n_neurons, n_edges, n_speeds, n_angles)

    theta = np.exp(1j * np.deg2rad(np.array(angles_deg, dtype=np.float32))).astype(np.complex64)

    columns = ["id", "type", "side"]
    for s in speeds:
        columns += [f"DSI_{s}", f"Angle_{s}"]
    columns += ["DSI_all", "Angle_all"]

    os.makedirs(side_out_dir, exist_ok=True)

    for ei, et in enumerate(EDGE_TYPES):
        print(f"\nComputing vectorized DSI: {et}")

        edge_matrix = response_4d[:, ei, :, :].astype(np.float32, copy=True)
        dsi_speed, angle_speed, dsi_all, angle_all = compute_dsi_for_edge_vectorized(
            edge_matrix=edge_matrix,
            theta_complex=theta,
            diff_threshold=DIFF_THRESHOLD
        )

        out_arr = np.empty((n_neurons, 2 * n_speeds + 2), dtype=np.float32)
        col = 0
        for si in range(n_speeds):
            out_arr[:, col] = dsi_speed[:, si]
            out_arr[:, col + 1] = angle_speed[:, si]
            col += 2
        out_arr[:, col] = dsi_all
        out_arr[:, col + 1] = angle_all

        df = pd.DataFrame(out_arr, columns=columns[3:])
        df.insert(0, "side", sides)
        df.insert(0, "type", types)
        df.insert(0, "id", all_neurons)
        df.sort_values(["type", "id"], inplace=True)

        out_csv = os.path.join(out_dir, f"DSI_all_speeds_{et}.csv")
        df.to_csv(out_csv, index=False)
        print(f"Saved: {out_csv}")

        for side in ["left", "right"]:
            df_side = df[df["side"] == side].copy()
            out_df = df_side[["type", "id", "DSI_all", "Angle_all"]].rename(
                columns={"DSI_all": "DSI", "Angle_all": "PreferredAngle"}
            )
            out_path = os.path.join(side_out_dir, f"DSI_{et}_{side}.csv")
            out_df.to_csv(out_path, index=False)
            print(f"Saved: {out_path}")


def main():
    print("=" * 70)
    print("Fast DSI peak-response pipeline")
    print("=" * 70)
    print(f"responses_dir: {responses_dir}")
    print(f"NUM_LOAD_WORKERS: {NUM_LOAD_WORKERS}")
    print(f"PEAK_THRESHOLD: {PEAK_THRESHOLD}")
    print(f"DIFF_THRESHOLD: {DIFF_THRESHOLD}")

    npz_files = sorted(glob.glob(os.path.join(responses_dir, "*.npz")))
    if not npz_files:
        raise FileNotFoundError(f"No npz files found in: {responses_dir}")

    print(f"Found npz files: {len(npz_files)}")

    entries = {
        (et, s, a): None
        for et in EDGE_TYPES
        for s in speeds
        for a in angles_deg
    }

    max_workers = min(NUM_LOAD_WORKERS, len(npz_files))
    with ProcessPoolExecutor(max_workers=max_workers) as executor:
        futures = [executor.submit(load_one_npz_peak, f) for f in npz_files]

        for fut in tqdm(as_completed(futures), total=len(futures), desc="Loading npz"):
            res = fut.result()
            if res is None:
                continue
            et, s, a, ids, peaks = res
            entries[(et, s, a)] = {
                "neuron_ids": ids,
                "peak_values": peaks
            }

    missing = [k for k, v in entries.items() if v is None]
    if missing:
        print("\nWarning: missing conditions:")
        for k in missing:
            print("  ", k)

    all_neurons, data_matrix = build_peak_matrix(entries)
    print(f"Number of neurons: {len(all_neurons):,}")
    print(f"Peak matrix shape: {data_matrix.shape}")

    _, _, types, sides = read_metadata(all_neurons)

    df_peak = save_wide_peak_csv(all_neurons, data_matrix, types, sides)
    save_min_removed_csv(df_peak, data_matrix)
    save_dsi_csvs(all_neurons, data_matrix, types, sides)

    print("\nAll finished successfully.")


if __name__ == "__main__":
    main()


Fast DSI peak-response pipeline
responses_dir: ../results/PRS/DSI/202606
NUM_LOAD_WORKERS: 32
PEAK_THRESHOLD: 0.0
DIFF_THRESHOLD: 0
Found npz files: 120


Loading npz: 100%|██████████| 120/120 [00:21<00:00,  5.47it/s]


Number of neurons: 139,255
Peak matrix shape: (139255, 120)
Peak responses CSV saved: ../results/DSI/peak_responses_all_wide.csv
Peak-min-removed CSV saved: ../results/DSI/peak_responses_all_wide_min_removed.csv

Computing vectorized DSI: ON
Saved: ../results/DSI/DSI_all_speeds_ON.csv
Saved: ../results/DSI/side_separated/DSI_ON_left.csv
Saved: ../results/DSI/side_separated/DSI_ON_right.csv

Computing vectorized DSI: OFF
Saved: ../results/DSI/DSI_all_speeds_OFF.csv
Saved: ../results/DSI/side_separated/DSI_OFF_left.csv
Saved: ../results/DSI/side_separated/DSI_OFF_right.csv

All finished successfully.
